# Hybrid RAG with BM25, MiniLM, and Qwen2.5-0.5B-Instruct

This notebook implements hybrid retrieval-augmented generation for a small corpus of country descriptions. Run it top to bottom in Google Colab (CPU works; a GPU makes the 0.5B model faster). The first run downloads `all-MiniLM-L6-v2` and `Qwen/Qwen2.5-0.5B-Instruct`.

The pipeline follows the six required stages:

1. Split each document into chunks of at most a few hundred tokens.
2. Build a scikit-learn TF-IDF encoding and a MiniLM embedding for every chunk.
3. Rank chunks with BM25 (`bm25s`) for exact lexical matches.
4. Rank chunks with cosine similarity over `all-MiniLM-L6-v2` embeddings.
5. Fuse the two rankings with reciprocal rank fusion and drop near-duplicate chunks.
6. Insert the top-K chunks into a prompt and generate the answer with `Qwen/Qwen2.5-0.5B-Instruct`.

Stack used here: scikit-learn, `sentence-transformers/all-MiniLM-L6-v2`, `bm25s`, the PyTorch / Transformers libraries those models require, and `Qwen/Qwen2.5-0.5B-Instruct`.

Section 11 is a student assignment on the failure this pipeline creates: the single best chunk often does not carry enough of the original document to answer.


## 0. Install the allowed libraries


In [1]:
import os
import subprocess
import sys

# Quiet, reproducible hub downloads. Set before Transformers or Sentence-Transformers import.
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

packages = [
    "scikit-learn>=1.3",
    "bm25s>=0.2.10",
    "sentence-transformers>=3.0",
    "transformers>=4.46",
    "accelerate>=0.34",
    "jinja2>=3.1.6",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Libraries ready:", ", ".join(packages))


Libraries ready: scikit-learn>=1.3, bm25s>=0.2.10, sentence-transformers>=3.0, transformers>=4.46, accelerate>=0.34, jinja2>=3.1.6


## 1. Settings

`all-MiniLM-L6-v2` truncates inputs at 256 WordPiece tokens, so every chunk stays under that window and under a few hundred tokens. These source files are only about 80–100 tokens each, so the budget below is tighter than the ceiling. A budget of 200 would leave each file as a single chunk. Raise `MAX_CHUNK_TOKENS` (up to 256) for a longer corpus.


In [2]:
import random
import re
import warnings
import numpy as np
import torch

warnings.filterwarnings("ignore", message="IProgress not found")

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MAX_CHUNK_TOKENS = 72
OVERLAP_SENTENCES = 1
LONG_SENTENCE_OVERLAP_TOKENS = 16

BM25_CANDIDATES = 8
SEMANTIC_CANDIDATES = 8
RRF_K = 60
TOP_K = 4
NEAR_DUP_JACCARD = 0.80

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_NEW_TOKENS = 160

CORPUS_URLS = [
    # Assignment link. Its query string is truncated (`raw=tru`), so the next URL is the fallback.
    "https://github.com/gakudo-ai/open-datasets/blob/main/asia_documents.zip?raw=tru",
    "https://github.com/gakudo-ai/open-datasets/raw/main/asia_documents.zip",
]

assert 8 <= MAX_CHUNK_TOKENS <= 256, "Chunk budget must stay within a few hundred tokens and MiniLM's window."
assert TOP_K >= 1
print(
    f"chunk budget={MAX_CHUNK_TOKENS} tokens | overlap sentences={OVERLAP_SENTENCES} | "
    f"top-k={TOP_K} | RRF k={RRF_K}"
)


chunk budget=72 tokens | overlap sentences=1 | top-k=4 | RRF k=60


## 2. Download the corpus

The archive holds one short `.txt` file per country. The loader accepts the assignment URL and, if that response is not a zip, the canonical raw GitHub URL.


In [3]:
import io
import zipfile
import urllib.request
from pathlib import Path

CORPUS_DIR = Path("asia_documents")


def download_corpus(urls, dest):
    dest.mkdir(parents=True, exist_ok=True)
    errors = []
    for url in urls:
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "hybrid-rag-colab"})
            with urllib.request.urlopen(request, timeout=60) as response:
                payload = response.read()
            if payload[:2] != b"PK":
                raise ValueError("response is not a zip archive")
            with zipfile.ZipFile(io.BytesIO(payload)) as archive:
                archive.extractall(dest)
            print(f"Downloaded corpus from {url}")
            return dest
        except Exception as exc:
            errors.append(f"{url} -> {exc}")
            print(f"Skipping {url}: {exc}")
    raise RuntimeError("Could not download the corpus.\n" + "\n".join(errors))


download_corpus(CORPUS_URLS, CORPUS_DIR)

documents = []
for path in sorted(CORPUS_DIR.rglob("*.txt")):
    text = path.read_text(encoding="utf-8").strip()
    if text:
        documents.append({"source": path.name, "text": text})

print(f"{len(documents)} documents")
for doc in documents:
    print(f"  {doc['source']:<18} {len(doc['text']):4d} characters")


Downloaded corpus from https://github.com/gakudo-ai/open-datasets/blob/main/asia_documents.zip?raw=tru
9 documents
  Indonesia.txt       415 characters
  Japan.txt           509 characters
  Malaysia.txt        387 characters
  Mongolia.txt        377 characters
  Philippines.txt     406 characters
  South_Korea.txt     390 characters
  Taiwan.txt          386 characters
  Thailand.txt        397 characters
  Vietnam.txt         414 characters


## 3. Chunk the documents

Chunks are packed from sentences. A sentence longer than the budget is split on MiniLM token ids, with a short token overlap, so a long paragraph still cannot exceed the cap. Neighboring chunks share one sentence when that overlap still fits in the budget. Token counts use the `all-MiniLM-L6-v2` WordPiece tokenizer, which is the tokenizer that will embed these strings.


In [4]:
from transformers import AutoTokenizer

chunk_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)


def token_count(text):
    return len(chunk_tokenizer.encode(text, add_special_tokens=False))


def split_sentences(text):
    return [part.strip() for part in re.split(r"(?<=[.!?])\s+", text.strip()) if part.strip()]


def split_overlong_sentence(sentence):
    ids = chunk_tokenizer.encode(sentence, add_special_tokens=False)
    if len(ids) <= MAX_CHUNK_TOKENS:
        return [sentence]
    pieces = []
    stride = max(1, MAX_CHUNK_TOKENS - LONG_SENTENCE_OVERLAP_TOKENS)
    start = 0
    while start < len(ids):
        end = min(start + MAX_CHUNK_TOKENS, len(ids))
        piece = ""
        while end > start:
            piece = chunk_tokenizer.decode(ids[start:end]).strip()
            if piece and token_count(piece) <= MAX_CHUNK_TOKENS:
                break
            end -= 1
        if not piece:
            break
        pieces.append(piece)
        if end >= len(ids):
            break
        start += stride
        if start >= end:
            start = end
    return pieces


def chunk_document(source, text):
    units = []
    for sentence in split_sentences(text):
        units.extend(split_overlong_sentence(sentence))

    raw_chunks = []
    current = []

    def count(parts):
        return token_count(" ".join(parts)) if parts else 0

    for unit in units:
        if current and count(current + [unit]) > MAX_CHUNK_TOKENS:
            raw_chunks.append(" ".join(current))
            overlap = current[-OVERLAP_SENTENCES:]
            current = overlap if count(overlap + [unit]) <= MAX_CHUNK_TOKENS else []
        current.append(unit)
    if current:
        raw_chunks.append(" ".join(current))

    records = []
    for chunk_id, chunk_text in enumerate(raw_chunks):
        records.append(
            {
                "source": source,
                "chunk_id": chunk_id,
                "text": chunk_text,
                "n_tokens": token_count(chunk_text),
            }
        )
    return records


chunks = []
for doc in documents:
    chunks.extend(chunk_document(doc["source"], doc["text"]))
for global_id, chunk in enumerate(chunks):
    chunk["id"] = global_id

assert chunks, "Chunking produced no text."
assert max(chunk["n_tokens"] for chunk in chunks) <= MAX_CHUNK_TOKENS

print(
    f"{len(documents)} documents -> {len(chunks)} chunks | "
    f"tokens min={min(c['n_tokens'] for c in chunks)} "
    f"max={max(c['n_tokens'] for c in chunks)} cap={MAX_CHUNK_TOKENS}"
)
for chunk in chunks:
    preview = chunk["text"][:110].replace("\n", " ")
    print(f"[{chunk['id']:>2}] {chunk['source']:<16} #{chunk['chunk_id']} ({chunk['n_tokens']:>2} tok) {preview}")


9 documents -> 18 chunks | tokens min=35 max=70 cap=72
[ 0] Indonesia.txt    #0 (64 tok) Indonesia is the world’s largest archipelago, consisting of over 17,000 islands, including Java, Sumatra, and 
[ 1] Indonesia.txt    #1 (37 tok) Bali is a popular tourist destination, famous for its temples, rice terraces, and beaches. Jakarta, the capita
[ 2] Japan.txt        #0 (66 tok) Japan is an island nation in East Asia, known for its unique blend of ancient traditions and cutting-edge tech
[ 3] Japan.txt        #1 (53 tok) Geographically, Japan consists of four main islands and numerous smaller ones, featuring mountains, forests, a
[ 4] Malaysia.txt     #0 (52 tok) Malaysia is a diverse country in Southeast Asia, divided into Peninsular Malaysia and Malaysian Borneo. Kuala 
[ 5] Malaysia.txt     #1 (45 tok) The country is known for its multicultural society, rainforests, and beautiful islands such as Langkawi. Malay
[ 6] Mongolia.txt     #0 (59 tok) Mongolia, while geographically in East As

## 4. TF-IDF encodings

`TfidfVectorizer` builds one sparse vector per chunk. Unigrams and bigrams keep names such as "petronas towers" and "mount fuji". `min_df=1` keeps terms that occur in a single chunk, which is where most of the proper nouns in this corpus live. Sublinear TF dampens repeated words inside a chunk.

These vectors are the lexical encoding required alongside the semantic embeddings. Exact-match retrieval in the hybrid ranker is BM25 (next sections). The helper below also lists TF-IDF nearest chunks so the encoding can be inspected on a query.


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

chunk_texts = [chunk["text"] for chunk in chunks]

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
)
tfidf_matrix = tfidf_vectorizer.fit_transform(chunk_texts)
feature_names = tfidf_vectorizer.get_feature_names_out()

print(
    f"TF-IDF matrix: {tfidf_matrix.shape[0]} chunks x {tfidf_matrix.shape[1]} terms, "
    f"{tfidf_matrix.nnz} non-zero weights"
)


def top_tfidf_terms(row_index, n=5):
    row = tfidf_matrix.getrow(row_index)
    if row.nnz == 0:
        return []
    order = row.data.argsort()[::-1][:n]
    return [(feature_names[row.indices[i]], float(row.data[i])) for i in order]


for chunk in chunks:
    terms = ", ".join(f"{term} ({weight:.2f})" for term, weight in top_tfidf_terms(chunk["id"]))
    print(f"[{chunk['id']:>2}] {chunk['source']:<16} {terms}")


def tfidf_search(query, top_n=5):
    query_vector = tfidf_vectorizer.transform([query])
    scores = cosine_similarity(query_vector, tfidf_matrix)[0]
    order = scores.argsort()[::-1][:top_n]
    return [(int(index), float(scores[index])) for index in order]


TF-IDF matrix: 18 chunks x 575 terms, 882 non-zero weights
[ 0] Indonesia.txt    bali (0.21), hinduism islam (0.14), islam indigenous (0.14), traditions bali (0.14), java sumatra (0.14)
[ 1] Indonesia.txt    modern historical (0.20), beaches jakarta (0.20), metropolis mix (0.20), bustling metropolis (0.20), capital bustling (0.20)
[ 2] Japan.txt        japan (0.19), ceremonies sumo (0.13), kabuki theater (0.13), heritage including (0.13), theater geographically (0.13)
[ 3] Japan.txt        temples modern (0.14), historical temples (0.14), experiences historical (0.14), kyoto osaka (0.14), distinct experiences (0.14)
[ 4] Malaysia.txt     malaysia (0.26), kuala lumpur (0.16), towers country (0.16), capital features (0.16), features iconic (0.16)
[ 5] Malaysia.txt     flavorful region (0.19), making flavorful (0.19), influences making (0.19), indian influences (0.19), chinese indian (0.19)
[ 6] Mongolia.txt     nomadic traditions (0.17), historical connection (0.17), connection genghis (

## 5. Semantic embeddings

`all-MiniLM-L6-v2` maps each chunk to a 384-dimensional sentence embedding. Vectors are L2-normalized, so cosine similarity is a dot product. Search still goes through `sklearn.metrics.pairwise.cosine_similarity`.


In [6]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBEDDING_MODEL)
chunk_embeddings = embedder.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
chunk_embeddings = np.asarray(chunk_embeddings, dtype=np.float32)

print(
    f"Embeddings: {chunk_embeddings.shape[0]} chunks x {chunk_embeddings.shape[1]} dims "
    f"from {EMBEDDING_MODEL}"
)


def semantic_search(query, top_n=SEMANTIC_CANDIDATES):
    top_n = min(top_n, len(chunks))
    query_embedding = embedder.encode(
        [query],
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    query_embedding = np.asarray(query_embedding, dtype=np.float32)
    scores = cosine_similarity(query_embedding, chunk_embeddings)[0]
    order = scores.argsort()[::-1][:top_n]
    return [(int(index), float(scores[index])) for index in order]


Embeddings: 18 chunks x 384 dims from sentence-transformers/all-MiniLM-L6-v2


## 6. BM25 exact-match retrieval

`bm25s` scores chunks with Lucene BM25 (`k1=1.5`, `b=0.75`) after the same style of tokenization scikit-learn uses: lowercase, English stopwords removed, tokens of at least two characters. A query is ranked by shared terms, which is the exact-match path. Candidate count is capped at the number of chunks because `bm25s` rejects a `k` larger than the index. Hits with a zero score are dropped; those slots are unused padding.


In [7]:
import bm25s

corpus_tokens = bm25s.tokenize(chunk_texts, stopwords="english", show_progress=False)
bm25 = bm25s.BM25(k1=1.5, b=0.75, method="lucene", backend="numpy")
bm25.index(corpus_tokens, show_progress=False)
print(f"BM25 index: {len(chunks)} chunks, vocabulary {len(bm25.vocab_dict)} tokens")


def bm25_search(query, top_n=BM25_CANDIDATES):
    top_n = min(top_n, len(chunks))
    if top_n == 0:
        return []
    query_tokens = bm25s.tokenize(query, stopwords="english", show_progress=False)
    try:
        documents, scores = bm25.retrieve(query_tokens, k=top_n, show_progress=False)
    except ValueError:
        return []
    hits = []
    seen = set()
    for index, score in zip(documents[0], scores[0]):
        index = int(index)
        score = float(score)
        # bm25s fills unused slots with score 0. Those slots are not matches.
        if score <= 0 or index in seen:
            continue
        seen.add(index)
        hits.append((index, score))
    return hits


BM25 index: 18 chunks, vocabulary 250 tokens


## 7. Reciprocal rank fusion and deduplication

BM25 scores and cosine scores are not on the same scale, so the lists are merged by rank. Reciprocal rank fusion (Cormack, Clarke, and Buettcher, 2009) scores a chunk as

`score(chunk) = Σ 1 / (k + rank)`

over the lists that contain it. `k` is 60. A chunk that both retrievers return accumulates both contributions, and each chunk appears once.

A second pass drops a lower-ranked chunk whose word Jaccard overlap with an already kept chunk is at least 0.80, so a sliding-window near copy does not fill the prompt.


In [8]:
def reciprocal_rank_fusion(rankings, k=RRF_K):
    scores = {}
    for ranking in rankings:
        for rank, index in enumerate(ranking, start=1):
            scores[index] = scores.get(index, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda item: (-item[1], item[0]))


def word_tokens(text):
    return set(re.findall(r"[a-z0-9]+", text.lower()))


def jaccard(left, right):
    if not left or not right:
        return 0.0
    return len(left & right) / len(left | right)


def deduplicate(ranked, top_k=TOP_K, threshold=NEAR_DUP_JACCARD):
    selected = []
    selected_tokens = []
    for index, score in ranked:
        tokens = word_tokens(chunks[index]["text"])
        if any(jaccard(tokens, kept) >= threshold for kept in selected_tokens):
            continue
        selected.append((index, score))
        selected_tokens.append(tokens)
        if len(selected) == top_k:
            break
    return selected


def hybrid_retrieve(query, top_k=TOP_K):
    lexical = bm25_search(query, BM25_CANDIDATES)
    semantic = semantic_search(query, SEMANTIC_CANDIDATES)
    fused = reciprocal_rank_fusion(
        [[index for index, _ in lexical], [index for index, _ in semantic]]
    )
    selected = deduplicate(fused, top_k=top_k)
    return {"bm25": lexical, "semantic": semantic, "fused": fused, "selected": selected}


# A chunk in both lists must outrank a chunk in only one list.
_demo = reciprocal_rank_fusion([[1, 2], [1, 3]])
assert [index for index, _ in _demo] == [1, 2, 3]
assert dict(_demo)[1] > dict(_demo)[2]
print("RRF check passed:", [(index, round(score, 5)) for index, score in _demo])


RRF check passed: [(1, 0.03279), (2, 0.01613), (3, 0.01613)]


## 8. Inspect retrieval before generation

The first query is an exact name, so BM25 and the embedding search agree. The second says "steppe people" rather than the file's wording ("vast steppes", "nomadic traditions"). BM25 then prefers a different country on generic words such as "people", while the embedding list and the fused list stay on Mongolia. TF-IDF neighbors are printed as a view of the encoding from section 4; they are not an input to fusion.


In [9]:
def format_hits(pairs, limit=5):
    lines = []
    for rank, (index, score) in enumerate(pairs[:limit], start=1):
        chunk = chunks[index]
        snippet = chunk["text"].replace("\n", " ")
        if len(snippet) > 140:
            snippet = snippet[:137] + "..."
        lines.append(
            f"{rank:>2}. score={score:.4f}  [{chunk['id']}] {chunk['source']}#{chunk['chunk_id']}  {snippet}"
        )
    return "\n".join(lines) if lines else "  (no hits)"


INSPECTION_QUERIES = [
    "Which capital is famous for the Petronas Towers?",
    "the steppe people and their capital city",
]

for query in INSPECTION_QUERIES:
    retrieved = hybrid_retrieve(query)
    print("=" * 88)
    print("Query:", query)
    print("\nBM25")
    print(format_hits(retrieved["bm25"]))
    print("\nSemantic")
    print(format_hits(retrieved["semantic"]))
    print("\nTF-IDF encoding neighbors")
    print(format_hits(tfidf_search(query)))
    print("\nFused top-K (these chunks go into the prompt)")
    print(format_hits(retrieved["selected"], limit=TOP_K))
    print()


def top_source(query):
    selected = hybrid_retrieve(query)["selected"]
    assert selected, f"No chunks retrieved for: {query}"
    return chunks[selected[0][0]]["source"]


expected_sources = {
    "Which capital is famous for the Petronas Towers?": "Malaysia.txt",
    "Where are the limestone karsts and floating markets?": "Vietnam.txt",
    "What is hanbok?": "South_Korea.txt",
    "Where is Mount Fuji?": "Japan.txt",
    "Which city is the capital of the nomadic steppe country?": "Mongolia.txt",
    "the steppe people and their capital city": "Mongolia.txt",
}
for query, source in expected_sources.items():
    got = top_source(query)
    assert got == source, f"{query!r} retrieved {got}, expected {source}"
    print(f"OK  {source:<16}  {query}")


Query: Which capital is famous for the Petronas Towers?

BM25
 1. score=2.3152  [4] Malaysia.txt#0  Malaysia is a diverse country in Southeast Asia, divided into Peninsular Malaysia and Malaysian Borneo. Kuala Lumpur, the capital, featur...
 2. score=0.5169  [1] Indonesia.txt#1  Bali is a popular tourist destination, famous for its temples, rice terraces, and beaches. Jakarta, the capital, is a bustling metropolis...
 3. score=0.4883  [13] Taiwan.txt#1  Taipei, the capital, is home to the towering Taipei 101 and the famous National Palace Museum. Taroko Gorge and Sun Moon Lake are among T...
 4. score=0.4397  [14] Thailand.txt#0  Thailand is a Southeast Asian country famous for its tropical beaches, ornate temples, and bustling street food culture. Bangkok, the cap...
 5. score=0.4060  [8] Philippines.txt#0  The Philippines is an archipelago of over 7,000 islands in Southeast Asia, known for its stunning beaches, vibrant festivals, and hospita...

Semantic
 1. score=0.5149  [4] Malaysi

## 9. Generate with the top-K chunks

The selected chunks are written into a single user message, each labeled with its rank, source file, and fusion score. `Qwen/Qwen2.5-0.5B-Instruct` is loaded in float16 on CUDA and float32 on CPU. Generation is greedy (`do_sample=False`), so repeated runs stay stable. The instruction tells the model to answer from the chunks, copy proper nouns as they are spelled, and cite the top chunk.


In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
if llm_tokenizer.pad_token_id is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, dtype=dtype)
llm.to(device)
llm.eval()
print(f"Loaded {LLM_MODEL} on {device} ({dtype})")


def build_messages(query, selected):
    blocks = []
    for rank, (index, score) in enumerate(selected, start=1):
        chunk = chunks[index]
        blocks.append(
            f"[{rank}] source={chunk['source']} chunk={chunk['chunk_id']} fusion_score={score:.4f}\n"
            f"{chunk['text']}"
        )
    context = "\n\n".join(blocks) if blocks else "(no chunks retrieved)"
    user = f"Context:\n{context}\n\nQuestion: {query}"
    return [
        {
            "role": "system",
            "content": (
                "Answer using only the context. "
                "Copy proper nouns exactly as spelled in the chunks. "
                "Cite [1]. Two sentences maximum."
            ),
        },
        {"role": "user", "content": user},
    ]


def generate_answer(query, selected, max_new_tokens=MAX_NEW_TOKENS):
    messages = build_messages(query, selected)
    prompt = llm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = llm_tokenizer(prompt, return_tensors="pt")
    inputs = {name: tensor.to(llm.device) for name, tensor in inputs.items()}
    with torch.inference_mode():
        generated = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=llm_tokenizer.pad_token_id,
        )
    new_ids = generated[0, inputs["input_ids"].shape[-1] :]
    return llm_tokenizer.decode(new_ids, skip_special_tokens=True).strip(), messages[1]["content"]


def ask(query, top_k=TOP_K):
    retrieved = hybrid_retrieve(query, top_k=top_k)
    answer, user_message = generate_answer(query, retrieved["selected"])
    print("=" * 88)
    print("Question:", query)
    print("\nPrompt context sent to the model:\n")
    print(user_message)
    print("\nAnswer:\n")
    print(answer)
    print()
    return answer


Loaded Qwen/Qwen2.5-0.5B-Instruct on cpu (torch.float32)


## 10. Ask the corpus

Each call retrieves with BM25 and MiniLM, fuses and deduplicates, then generates from the top-K chunks. The cell prints the prompt that was sent, then the answer. This 0.5B model often replies with the shortest fitting name, so the prompt is the full evidence. Edit the list or call `ask("...")` in a new cell for another question.


In [11]:
questions = [
    "Which capital is famous for the Petronas Towers?",
    "Where are the limestone karsts and floating markets?",
    "What is hanbok, and which palace is in that country's capital?",
    "What is Japan's most iconic mountain, and which major cities does the corpus name?",
    "Which city is the capital of the nomadic steppe country?",
]

answers = {question: ask(question) for question in questions}


Question: Which capital is famous for the Petronas Towers?

Prompt context sent to the model:

Context:
[1] source=Malaysia.txt chunk=0 fusion_score=0.0328
Malaysia is a diverse country in Southeast Asia, divided into Peninsular Malaysia and Malaysian Borneo. Kuala Lumpur, the capital, features the iconic Petronas Towers. The country is known for its multicultural society, rainforests, and beautiful islands such as Langkawi.

[2] source=Indonesia.txt chunk=1 fusion_score=0.0320
Bali is a popular tourist destination, famous for its temples, rice terraces, and beaches. Jakarta, the capital, is a bustling metropolis with a mix of modern and historical sites.

[3] source=Taiwan.txt chunk=1 fusion_score=0.0320
Taipei, the capital, is home to the towering Taipei 101 and the famous National Palace Museum. Taroko Gorge and Sun Moon Lake are among Taiwan’s most stunning natural attractions.

[4] source=Philippines.txt chunk=0 fusion_score=0.0310
The Philippines is an archipelago of over 7,000 i

Question: Where are the limestone karsts and floating markets?

Prompt context sent to the model:

Context:
[1] source=Vietnam.txt chunk=1 fusion_score=0.0328
Ha Long Bay’s limestone karsts and the Mekong Delta’s floating markets are famous geographical highlights. Vietnamese culture is deeply influenced by Confucian values, French colonial heritage, and indigenous traditions.

[2] source=Vietnam.txt chunk=0 fusion_score=0.0320
Vietnam is a Southeast Asian country known for its rich history, diverse landscapes, and delicious cuisine. Hanoi and Ho Chi Minh City are its major urban centers, each with a unique character. Ha Long Bay’s limestone karsts and the Mekong Delta’s floating markets are famous geographical highlights.

[3] source=Taiwan.txt chunk=0 fusion_score=0.0308
Taiwan is an island located off the southeastern coast of China, known for its vibrant night markets, lush mountains, and rich history. It blends Chinese, Japanese, and indigenous influences in its culture. Taipei, t

Question: What is hanbok, and which palace is in that country's capital?

Prompt context sent to the model:

Context:
[1] source=South_Korea.txt chunk=0 fusion_score=0.0328
South Korea is a dynamic country in East Asia, blending modern urban life with deep-rooted traditions. Seoul, the capital, is a global technology hub and home to historical landmarks like Gyeongbokgung Palace. The country is famous for K-pop, Korean BBQ, and hanbok (traditional clothing).

[2] source=South_Korea.txt chunk=1 fusion_score=0.0323
The country is famous for K-pop, Korean BBQ, and hanbok (traditional clothing). Natural attractions include Jeju Island’s volcanic landscapes and the scenic countryside of Boseong.

[3] source=Thailand.txt chunk=0 fusion_score=0.0315
Thailand is a Southeast Asian country famous for its tropical beaches, ornate temples, and bustling street food culture. Bangkok, the capital, is known for its vibrant nightlife and historical sites like the Grand Palace and Wat Arun.

[4] source=

Question: What is Japan's most iconic mountain, and which major cities does the corpus name?

Prompt context sent to the model:

Context:
[1] source=Japan.txt chunk=1 fusion_score=0.0328
Geographically, Japan consists of four main islands and numerous smaller ones, featuring mountains, forests, and a rugged coastline. Mount Fuji is its most iconic peak. Major cities include Tokyo, Kyoto, and Osaka, each offering distinct experiences from historical temples to modern skyscrapers.

[2] source=Japan.txt chunk=0 fusion_score=0.0323
Japan is an island nation in East Asia, known for its unique blend of ancient traditions and cutting-edge technology. It has a rich cultural heritage, including tea ceremonies, sumo wrestling, and kabuki theater. Geographically, Japan consists of four main islands and numerous smaller ones, featuring mountains, forests, and a rugged coastline.

[3] source=Taiwan.txt chunk=1 fusion_score=0.0312
Taipei, the capital, is home to the towering Taipei 101 and the famou

Question: Which city is the capital of the nomadic steppe country?

Prompt context sent to the model:

Context:
[1] source=Mongolia.txt chunk=0 fusion_score=0.0328
Mongolia, while geographically in East Asia, has a culture that is distinct from its neighbors. It is known for its vast steppes, nomadic traditions, and historical connection to Genghis Khan. Ulaanbaatar, the capital, serves as the country’s cultural and economic center.

[2] source=Mongolia.txt chunk=1 fusion_score=0.0318
Ulaanbaatar, the capital, serves as the country’s cultural and economic center. The Naadam Festival, featuring horse racing, wrestling, and archery, showcases Mongolia’s rich heritage.

[3] source=South_Korea.txt chunk=0 fusion_score=0.0313
South Korea is a dynamic country in East Asia, blending modern urban life with deep-rooted traditions. Seoul, the capital, is a global technology hub and home to historical landmarks like Gyeongbokgung Palace. The country is famous for K-pop, Korean BBQ, and hanbok (tr

## 11. Assignment: one chunk is not enough context

Retrieval returns a piece of a document. That piece can be the right piece and still leave out the sentence the question needs. This section is the assignment. Run the demonstration, then fill in the write-up.

Three gaps show up in this corpus.

**Split facts.** The chunker placed two details in different chunks. Malaysia's Petronas Towers are in chunk 0. The sentence that names Malay, Chinese, and Indian influences is in chunk 1. The overlapping sentence is the one about Langkawi, so the overlap keeps a shared sentence and still leaves each required fact in only one chunk.

**A missing name.** Indonesia chunk 1 describes Bali and then "Jakarta, the capital." The word Indonesia is not in that chunk. The index still stores `source=Indonesia.txt`, and section 9 copies that filename into the prompt. A reader of the chunk text does not have the country name. A reader of the prompt does, because the filename was attached beside the text.

**Parametric fill-in.** This model was pretrained on widely known geography. Given only the chunk that says Indonesia has 17,000 islands, it can still answer "Jakarta." That word is not in the chunk. Treat an answer as evidence about the chunk when the fact's own words appear in the text you supplied.

The demonstration searches the chunk text first, then prints the model's answer beside that check.


In [12]:
def fact_present(text, fact):
    return fact.lower() in text.lower()


def chunks_for_source(source):
    return [chunk for chunk in chunks if chunk["source"] == source]


def mentions_source(chunk):
    """True when the chunk text itself contains the country name from the filename."""
    stem = chunk["source"].replace(".txt", "").replace("_", " ").lower()
    text = chunk["text"].lower()
    if stem in text:
        return True
    tokens = [token for token in stem.split() if len(token) > 3]
    return any(token in text for token in tokens)


def unnamed_chunks():
    return [chunk for chunk in chunks if not mentions_source(chunk)]


def print_fact_map(source, facts):
    print(f"Source file: {source}")
    for chunk in chunks_for_source(source):
        flags = ", ".join(
            f"{fact!r} {'IN CHUNK' if fact_present(chunk['text'], fact) else 'MISSING'}"
            for fact in facts
        )
        print(f"  [{chunk['id']:>2}] #{chunk['chunk_id']} ({chunk['n_tokens']:>2} tok) {flags}")
        print(f"       {chunk['text']}")
    print()


def coverage(question, facts, top_k=TOP_K):
    """Show which required phrases are in the top chunk, and which are in the top-K union."""
    retrieved = hybrid_retrieve(question, top_k=top_k)
    selected = retrieved["selected"]
    top = chunks[selected[0][0]]
    union = "\n".join(chunks[index]["text"] for index, _ in selected)
    print("Question:", question)
    print(f"Top chunk [{top['id']}] {top['source']}#{top['chunk_id']}")
    for fact in facts:
        print(f"  top chunk     {fact!r}: {'IN CHUNK' if fact_present(top['text'], fact) else 'MISSING'}")
    for fact in facts:
        print(f"  top-{len(selected)} union  {fact!r}: {'IN CHUNK' if fact_present(union, fact) else 'MISSING'}")
    print("Chunks written into a top-K prompt:")
    for rank, (index, score) in enumerate(selected, start=1):
        chunk = chunks[index]
        print(f"  [{rank}] {chunk['source']}#{chunk['chunk_id']} score={score:.4f}")
    print()
    return retrieved


def expand_with_neighbors(selected):
    """Repair a split by adding the previous and next chunk from the same file.

    This brings back a fact the retriever never ranked. It does not help when the
    missing fact is more than one chunk away, and a high overlap can still make
    the neighbor look like a near-duplicate.
    """
    expanded = []
    seen = set()
    for index, score in selected:
        family = [chunk["id"] for chunk in chunks if chunk["source"] == chunks[index]["source"]]
        position = family.index(index)
        for neighbor in family[max(0, position - 1) : position + 2]:
            if neighbor in seen:
                continue
            seen.add(neighbor)
            expanded.append((neighbor, score if neighbor == index else 0.0))
    return expanded


def answer_from_passages(question, passages):
    """Generate from raw passage strings, with no filename attached."""
    blocks = [f"[{rank}]\n{passage}" for rank, passage in enumerate(passages, start=1)]
    user = "Context:\n" + "\n\n".join(blocks) + f"\n\nQuestion: {question}"
    messages = [
        {
            "role": "system",
            "content": (
                "Answer using only the context. "
                "Copy proper nouns exactly as spelled in the chunks. "
                "Cite [1]. Two sentences maximum."
            ),
        },
        {"role": "user", "content": user},
    ]
    prompt = llm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = llm_tokenizer(prompt, return_tensors="pt")
    inputs = {name: tensor.to(llm.device) for name, tensor in inputs.items()}
    with torch.inference_mode():
        generated = llm.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=llm_tokenizer.pad_token_id,
        )
    new_ids = generated[0, inputs["input_ids"].shape[-1] :]
    return llm_tokenizer.decode(new_ids, skip_special_tokens=True).strip()


MALAYSIA_QUESTION = "What cuisines shape Malaysian food, and which towers are in its capital?"
MALAYSIA_FACTS = ["Petronas", "Malay, Chinese, and Indian"]
INDONESIA_QUESTION = "How many islands make up Indonesia, and what is its capital?"
INDONESIA_FACTS = ["17,000", "Jakarta"]

print("=" * 88)
print("Split facts inside one file")
print("=" * 88)
print_fact_map("Malaysia.txt", MALAYSIA_FACTS)
malaysia_hits = coverage(MALAYSIA_QUESTION, MALAYSIA_FACTS)

malaysia_chunks = chunks_for_source("Malaysia.txt")
towers_only = [
    chunk["text"]
    for chunk in malaysia_chunks
    if fact_present(chunk["text"], "Petronas")
    and not fact_present(chunk["text"], "Malay, Chinese, and Indian")
]
both_malaysia = [chunk["text"] for chunk in malaysia_chunks]
print("Answer from the towers chunk alone:")
malaysia_one = answer_from_passages(MALAYSIA_QUESTION, towers_only)
print(malaysia_one)
print("\nAnswer from both Malaysia chunks:")
malaysia_both = answer_from_passages(MALAYSIA_QUESTION, both_malaysia)
print(malaysia_both)
print()

assert fact_present(towers_only[0], "Petronas")
assert not fact_present(towers_only[0], "Malay, Chinese, and Indian")
assert fact_present("\n".join(both_malaysia), "Malay, Chinese, and Indian")
assert "malay, chinese, and indian" not in malaysia_one.lower()
assert "malay, chinese, and indian" in malaysia_both.lower()

print("=" * 88)
print("The top chunk can miss a fact stored in another chunk of the same file")
print("=" * 88)
print_fact_map("Indonesia.txt", INDONESIA_FACTS)
indonesia_hits = coverage(INDONESIA_QUESTION, INDONESIA_FACTS)
indonesia_top = chunks[indonesia_hits["selected"][0][0]]
indonesia_union = "\n".join(chunks[index]["text"] for index, _ in indonesia_hits["selected"])
indonesia_answer = answer_from_passages(INDONESIA_QUESTION, [indonesia_top["text"]])
print("Answer from the top chunk alone:")
print(indonesia_answer)
print(f"Top chunk contains 'Jakarta': {fact_present(indonesia_top['text'], 'Jakarta')}")
print(f"Model answer contains 'Jakarta': {'jakarta' in indonesia_answer.lower()}")
jakarta_in_union = fact_present(indonesia_union, "Jakarta")
print(f"Top-{len(indonesia_hits['selected'])} union contains 'Jakarta': {jakarta_in_union}")
if jakarta_in_union:
    print(
        "This run retrieved the Jakarta chunk inside the top-K. "
        "The top chunk still does not contain that word. "
        "Reciprocal rank fusion scores near the bottom of the list are close, "
        "so another environment can rank that chunk just outside the top-K."
    )
else:
    print(
        "This run left the Jakarta chunk outside the top-K. "
        "Taking more hits did not restore the capital, because the chunk that names it was ranked too low."
    )

expanded = expand_with_neighbors(indonesia_hits["selected"][:1])
expanded_text = "\n".join(chunks[index]["text"] for index, _ in expanded)
print("\nNeighbor expansion of that top chunk brings the adjacent chunk back:")
for index, _ in expanded:
    chunk = chunks[index]
    print(f"  [{chunk['id']}] {chunk['source']}#{chunk['chunk_id']}")
print(f"Expanded context contains '17,000': {fact_present(expanded_text, '17,000')}")
print(f"Expanded context contains 'Jakarta': {fact_present(expanded_text, 'Jakarta')}")
print()

assert fact_present(indonesia_top["text"], "17,000")
assert not fact_present(indonesia_top["text"], "Jakarta")
assert fact_present(expanded_text, "17,000")
assert fact_present(expanded_text, "Jakarta")

print("=" * 88)
print("A chunk that lost the country name")
print("=" * 88)
for chunk in unnamed_chunks():
    print(f"[{chunk['id']}] {chunk['source']}#{chunk['chunk_id']}")
    print(chunk["text"])
    print()

jakarta_chunk = next(
    chunk for chunk in chunks if chunk["source"] == "Indonesia.txt" and chunk["chunk_id"] == 1
)
assert "indonesia" not in jakarta_chunk["text"].lower()
name_question = "Which country is Jakarta the capital of?"
plain = jakarta_chunk["text"]
headed = f"Document: {jakarta_chunk['source']}\n{plain}"
print("Country name inside the chunk text:", "indonesia" in plain.lower())
print("Country name after the filename is written in:", "indonesia" in headed.lower())
print("Same passage, no filename:")
print(answer_from_passages(name_question, [plain]))
print("\nSame passage, with the filename written into the prompt:")
print(answer_from_passages(name_question, [headed]))
print(
    "The answers can match. The filename changes the passage. "
    "A matching answer means the model already supplied the country name from memory."
)
print()
print("Assignment checks passed.")


Split facts inside one file
Source file: Malaysia.txt
  [ 4] #0 (52 tok) 'Petronas' IN CHUNK, 'Malay, Chinese, and Indian' MISSING
       Malaysia is a diverse country in Southeast Asia, divided into Peninsular Malaysia and Malaysian Borneo. Kuala Lumpur, the capital, features the iconic Petronas Towers. The country is known for its multicultural society, rainforests, and beautiful islands such as Langkawi.
  [ 5] #1 (45 tok) 'Petronas' MISSING, 'Malay, Chinese, and Indian' IN CHUNK
       The country is known for its multicultural society, rainforests, and beautiful islands such as Langkawi. Malaysian cuisine blends Malay, Chinese, and Indian influences, making it one of the most flavorful in the region.

Question: What cuisines shape Malaysian food, and which towers are in its capital?
Top chunk [4] Malaysia.txt#0
  top chunk     'Petronas': IN CHUNK
  top chunk     'Malay, Chinese, and Indian': MISSING
  top-4 union  'Petronas': IN CHUNK
  top-4 union  'Malay, Chinese, and Indian': 

Cuisines shape Malaysian food. The iconic Petronas Towers are in its capital.

Answer from both Malaysia chunks:


Malaysian cuisine blends Malay, Chinese, and Indian influences, making it one of the most flavorful in the region. Kuala Lumpur's iconic Petronas Towers feature the iconic Petronas Towers.

The top chunk can miss a fact stored in another chunk of the same file
Source file: Indonesia.txt
  [ 0] #0 (64 tok) '17,000' IN CHUNK, 'Jakarta' MISSING
       Indonesia is the world’s largest archipelago, consisting of over 17,000 islands, including Java, Sumatra, and Bali. The country is known for its rich cultural diversity, with influences from Hinduism, Islam, and indigenous traditions. Bali is a popular tourist destination, famous for its temples, rice terraces, and beaches.
  [ 1] #1 (37 tok) '17,000' MISSING, 'Jakarta' IN CHUNK
       Bali is a popular tourist destination, famous for its temples, rice terraces, and beaches. Jakarta, the capital, is a bustling metropolis with a mix of modern and historical sites.

Question: How many islands make up Indonesia, and what is its capital?
Top chu

Answer from the top chunk alone:
Indonesia has 17,000 islands, and its capital is Jakarta.
Top chunk contains 'Jakarta': False
Model answer contains 'Jakarta': True
Top-4 union contains 'Jakarta': False
This run left the Jakarta chunk outside the top-K. Taking more hits did not restore the capital, because the chunk that names it was ranked too low.

Neighbor expansion of that top chunk brings the adjacent chunk back:
  [0] Indonesia.txt#0
  [1] Indonesia.txt#1
Expanded context contains '17,000': True
Expanded context contains 'Jakarta': True

A chunk that lost the country name
[1] Indonesia.txt#1
Bali is a popular tourist destination, famous for its temples, rice terraces, and beaches. Jakarta, the capital, is a bustling metropolis with a mix of modern and historical sites.

[9] Philippines.txt#1
The country is famous for destinations like Boracay, Palawan, and the Chocolate Hills of Bohol. Filipino culture is influenced by Malay, Spanish, and American traditions.

Country name inside

Jakarta is the capital of Indonesia.

Same passage, with the filename written into the prompt:


Jakarta is the capital of Indonesia.
The answers can match. The filename changes the passage. A matching answer means the model already supplied the country name from memory.

Assignment checks passed.


### Tasks

Hand in the write-up below. The code cell under it runs a Japan starter so the notebook executes before you edit it. Replace that starter with your own question.

**A. Malaysia.** From the printout, say which chunk holds the towers and which holds the cuisine. Say why the overlapping Langkawi sentence did not put both facts in one chunk.

**B. Your own split-fact question.** Use a file other than `Malaysia.txt` and `Indonesia.txt`. Pick two phrases that live in different `chunk_id` values of that file, and write a question that needs both. Run `coverage`. Record whether the top chunk contains each phrase, and whether the top-K union does. If the model names a phrase the coverage line marks MISSING, that name came from the model, not from the chunk.

Places worth checking: kabuki theater and Mount Fuji; Genghis Khan and the Naadam Festival; night markets and Taroko Gorge.

**C. The unnamed chunk.** `unnamed_chunks()` lists chunks whose text does not contain the country name. For Indonesia chunk 1, write two facts a reader can get from the text alone, and one fact they can get only when the prompt also shows `source=Indonesia.txt`.

**D. Repair one gap, and name the gap you left open.** `expand_with_neighbors` adds the adjacent chunk from the same file. A document header writes the filename into the prompt and leaves every other fact where it was.

Run one of those on your Task B question. In a short paragraph, say which gap closed, which gap stayed open, and why the Indonesia answer is allowed to say "Jakarta" only when that word is in the supplied text.

| Part | A complete answer |
| --- | --- |
| A | Names both Malaysia chunks and the overlap sentence that failed to carry either required fact. |
| B | A new question, two phrases in different chunks, and a correct in-chunk / missing call for the top chunk and the top-K union. |
| C | Two facts supported by Indonesia chunk 1 alone, and one fact that depends on the filename. |
| D | One repair, rerun, and a paragraph that keeps split facts, missing names, and model memory separate. |

### Your write-up

**A. Malaysia split.**

**B. My question.**

Question:

Phrases:

Top chunk:

Top-K union:

**C. Unnamed chunk.**

**D. Repair.**

Function I ran:

Gap it closed:

Gap it left open:


In [13]:
# Starter for Task B and Task D. Replace MY_QUESTION and MY_FACTS with your own split.
# "kabuki" is only in Japan chunk 0. "Mount Fuji" is only in Japan chunk 1.
MY_QUESTION = "What traditional arts does Japan have, and what is its most iconic peak?"
MY_FACTS = ["kabuki", "Mount Fuji"]
MY_SOURCE = "Japan.txt"

print_fact_map(MY_SOURCE, MY_FACTS)
my_hits = coverage(MY_QUESTION, MY_FACTS)
print("Answer from the top chunk alone:")
print(answer_from_passages(MY_QUESTION, [chunks[my_hits["selected"][0][0]]["text"]]))
print("\nAnswer after neighbor expansion of the top chunk:")
my_expanded = expand_with_neighbors(my_hits["selected"][:1])
for index, _ in my_expanded:
    chunk = chunks[index]
    print(f"  [{chunk['id']}] {chunk['source']}#{chunk['chunk_id']}")
print(answer_from_passages(MY_QUESTION, [chunks[index]["text"] for index, _ in my_expanded]))
print()
print("Chunks whose text does not name the source country:")
for chunk in unnamed_chunks():
    print(f"  [{chunk['id']}] {chunk['source']}#{chunk['chunk_id']}")


Source file: Japan.txt
  [ 2] #0 (66 tok) 'kabuki' IN CHUNK, 'Mount Fuji' MISSING
       Japan is an island nation in East Asia, known for its unique blend of ancient traditions and cutting-edge technology. It has a rich cultural heritage, including tea ceremonies, sumo wrestling, and kabuki theater. Geographically, Japan consists of four main islands and numerous smaller ones, featuring mountains, forests, and a rugged coastline.
  [ 3] #1 (53 tok) 'kabuki' MISSING, 'Mount Fuji' IN CHUNK
       Geographically, Japan consists of four main islands and numerous smaller ones, featuring mountains, forests, and a rugged coastline. Mount Fuji is its most iconic peak. Major cities include Tokyo, Kyoto, and Osaka, each offering distinct experiences from historical temples to modern skyscrapers.

Question: What traditional arts does Japan have, and what is its most iconic peak?
Top chunk [3] Japan.txt#1
  top chunk     'kabuki': MISSING
  top chunk     'Mount Fuji': IN CHUNK
  top-4 union  'kab

Traditional arts include tea ceremony, Kabuki theater, and calligraphy. Its most iconic peak is Mount Fuji.

Answer after neighbor expansion of the top chunk:
  [2] Japan.txt#0
  [3] Japan.txt#1


Two sentences maximum.
Traditional arts that Japan has include tea ceremonies, sumo wrestling, and kabuki theater. Its most iconic peak is Mount Fuji.

Chunks whose text does not name the source country:
  [1] Indonesia.txt#1
  [9] Philippines.txt#1


## Knobs

| Setting | Role |
| --- | --- |
| `MAX_CHUNK_TOKENS` | Hard cap per chunk. Keep it at or below a few hundred, and at or below 256 for this embedding model. |
| `OVERLAP_SENTENCES` | Sentences repeated at the start of the next chunk when they still fit. |
| `BM25_CANDIDATES`, `SEMANTIC_CANDIDATES` | Depth of each list before fusion. |
| `RRF_K` | Rank constant in `1 / (k + rank)`. Larger values flatten the difference between rank 1 and rank 2. |
| `TOP_K` | How many fused chunks are written into the Qwen prompt. |
| `NEAR_DUP_JACCARD` | Overlap at which a lower-ranked chunk is treated as a duplicate of one already kept. |
